# dcgan-normal-init-002 — ex2: named-module DCGAN init with per-type gain and a report

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dcgan-normal-init-002`. Running the final beacon cell reports progress against the `GAN: DCGAN normal init 0.02` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: DCGAN normal init 0.02` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dcgan-normal-init-002`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dcgan-normal-init-002"
DD_SUBTOPIC = "GAN: DCGAN normal init 0.02"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## DCGAN normal init — named-module variant

Ex1 walked layers via `model.apply` (a pure transform).  Here we want a **report**: which named submodule got which init, by dotted path. The tool is `model.named_modules()` — same recursion as `modules()`, but each yielded item is `(qualified_name: str, module)`:

```python
for name, m in model.named_modules():
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, mean=0.0, std=0.02)
        records.append((name, 'conv', m.weight.std().item()))
```

**Why named_modules over modules.** When you need to LOG what you initialized (which is most real-world training-script init code), you want the qualified name (`'features.3.conv'`) — not just the type. `apply` gives you neither.

**Why also gain-scale.** Production DCGAN code sometimes scales the std by a `gain` factor per layer type (e.g. ConvTranspose gets `gain` × 0.02 to compensate for the upsample). The ex2 drills the parametric form.

### Exercise 2 — named-module DCGAN init with per-type gain and a report

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze a model by walking `model.named_modules()`, apply gain-scaled `N(0, gain*0.02)` init to Conv/ConvTranspose layers (gain per type), and return a sorted report of `(qualified_name, type_label, post_init_std)`.
> Keywords: dcgan, init, named-modules, gain, report
> ```

**KCs targeted:** `named-modules-iter-with-qname`, `gain-scaled-normal-init`

Implement `ex2_named_dcgan_init(model, conv_gain=1.0, convt_gain=1.0)`. Three responsibilities:

1. Walk `model.named_modules()`. SKIP the root entry (qname == `''`).
2. For each yielded `(qname, m)`:
   - If `isinstance(m, nn.Conv2d)`: call `nn.init.normal_(m.weight, 0.0, conv_gain * 0.02)`, then append `(qname, 'conv2d', m.weight.std().item())` to a `records` list.
   - elif `isinstance(m, nn.ConvTranspose2d)`: call `nn.init.normal_(m.weight, 0.0, convt_gain * 0.02)`, then append `(qname, 'convtranspose2d', m.weight.std().item())`.
   - Other layer types: skip (no init, no record).
3. Return `sorted(records)` — sorted by qname (lexicographic, default tuple sort).

Input: `model` — `nn.Module`; `conv_gain`, `convt_gain` — floats, default 1.0.
Output: `list[tuple[str, str, float]]`.

The visualization plots the per-layer post-init std as a bar chart with two colors (conv vs convt), showing how the gain knobs shift the std away from the baseline 0.02.

In [ ]:
def ex2_named_dcgan_init(model: nn.Module, conv_gain: float = 1.0, convt_gain: float = 1.0) -> list:
    """Init Conv/ConvT via named_modules; return sorted report of (qname, type, std)."""
    raise NotImplementedError()


def _test_ex2():
    import torch.nn as nn

    # Build a model with named submodules so qnames are non-trivial.
    class Block(nn.Module):
        def __init__(self, ic, oc):
            super().__init__()
            self.conv = nn.Conv2d(ic, oc, 3, padding=1)
            self.bn = nn.BatchNorm2d(oc)
            self.up = nn.ConvTranspose2d(oc, oc, 4, stride=2, padding=1)

    model = nn.Sequential()
    model.add_module('block_a', Block(3, 8))
    model.add_module('block_b', Block(8, 16))
    model.add_module('head', nn.Linear(16, 10))

    bn_w_before = model.block_a.bn.weight.detach().clone()
    lin_w_before = model.head.weight.detach().clone()

    # Run with default gains (1.0, 1.0) — std should be ~0.02 for both types.
    report = ex2_named_dcgan_init(model)
    assert isinstance(report, list), f'expected list, got {type(report).__name__}'
    assert len(report) == 4, f'expected 4 records (2 conv + 2 convT), got {len(report)}: {report}'

    # Records sorted by qname.
    qnames = [r[0] for r in report]
    assert qnames == sorted(qnames), f'records not sorted by qname: {qnames}'

    # All expected qnames present.
    expected_qnames = {'block_a.conv', 'block_a.up', 'block_b.conv', 'block_b.up'}
    assert set(qnames) == expected_qnames, f'qnames wrong: {set(qnames)} vs {expected_qnames}'

    # Type labels correct.
    for qname, tlabel, std in report:
        if 'conv' == qname.split('.')[-1]:
            assert tlabel == 'conv2d', f'{qname}: expected conv2d, got {tlabel}'
        elif 'up' == qname.split('.')[-1]:
            assert tlabel == 'convtranspose2d', f'{qname}: expected convtranspose2d, got {tlabel}'
        assert abs(std - 0.02) < 0.012, f'{qname}: std {std:.5f} not ~0.02'

    # BatchNorm + Linear untouched.
    assert t.equal(model.block_a.bn.weight, bn_w_before), 'BN must be untouched'
    assert t.equal(model.head.weight, lin_w_before), 'Linear must be untouched'

    # Gain knobs widen the per-type std.
    model2 = nn.Sequential()
    model2.add_module('block_a', Block(3, 8))
    model2.add_module('block_b', Block(8, 16))
    report2 = ex2_named_dcgan_init(model2, conv_gain=3.0, convt_gain=5.0)
    conv_stds = [s for _, tl, s in report2 if tl == 'conv2d']
    convt_stds = [s for _, tl, s in report2 if tl == 'convtranspose2d']
    for s in conv_stds:
        assert abs(s - 0.06) < 0.025, f'conv std with gain=3 should be ~0.06, got {s:.5f}'
    for s in convt_stds:
        assert abs(s - 0.10) < 0.04, f'convT std with gain=5 should be ~0.10, got {s:.5f}'

    # Root entry must be skipped — no record with qname == ''.
    assert all(r[0] != '' for r in report), 'root qname must be filtered out'

    # Empty model (no Conv/ConvT) → empty report.
    bare = nn.Sequential(nn.Linear(4, 4), nn.BatchNorm1d(4))
    assert ex2_named_dcgan_init(bare) == [], 'model with no conv layers should give empty report'

    # --- Visualization: per-layer std bar chart with type colors ---
    viz = nn.Sequential()
    viz.add_module('block_a', Block(3, 16))
    viz.add_module('block_b', Block(16, 32))
    viz.add_module('block_c', Block(32, 64))
    report_viz = ex2_named_dcgan_init(viz, conv_gain=2.0, convt_gain=4.0)
    colors = ['steelblue' if tl == 'conv2d' else 'coral' for _, tl, _ in report_viz]
    labels = [q for q, _, _ in report_viz]
    vals = [s for _, _, s in report_viz]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(range(len(vals)), vals, color=colors, edgecolor='black')
    ax.axhline(0.02, ls='--', color='gray', label='baseline std=0.02')
    ax.axhline(0.04, ls=':', color='steelblue', label='conv (gain=2) → 0.04')
    ax.axhline(0.08, ls=':', color='coral', label='convT (gain=4) → 0.08')
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_ylabel('post-init std'); ax.set_title('DCGAN named-module init — per-type gain')
    ax.legend(); plt.tight_layout()
    plt.show()
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_named_dcgan_init(model: nn.Module, conv_gain: float = 1.0, convt_gain: float = 1.0) -> list:
    records = []
    for qname, m in model.named_modules():
        if qname == '':
            continue
        if isinstance(m, nn.Conv2d):
            nn.init.normal_(m.weight, 0.0, conv_gain * 0.02)
            records.append((qname, 'conv2d', m.weight.std().item()))
        elif isinstance(m, nn.ConvTranspose2d):
            nn.init.normal_(m.weight, 0.0, convt_gain * 0.02)
            records.append((qname, 'convtranspose2d', m.weight.std().item()))
    return sorted(records)
```

**`named_modules()` over `apply`.** `apply` gives you the module but not its name — fine for pure init, bad for logging. The named form is the canonical pattern in training scripts that emit per-layer stats to wandb or tensorboard.

**Filter root via `qname == ''`.** The first item from `named_modules()` is always the root model — empty qname, model itself as the module. Forgetting to skip it doesn't break the init (the model itself isn't a Conv2d) but it appears as a stray in any qname-keyed dict.

**Gain as a multiplier, not a replacement.** Multiplying 0.02 by a gain factor lets you keep the DCGAN base std while tuning per type. Common in production: `gain=1.4` on the final ConvT to compensate for the Tanh activation.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()